# Observability with OpenTelemetry

Pixeltable can export its internal telemetry as OpenTelemetry (OTel) spans and metrics, so you can watch
operations in any OTel backend. This notebook wires the OTel bridge to **Grafana Cloud** and traces an
`insert()`.

An insert emits a nested span tree:

```
pixeltable.insert            (operation span, root)
└─ pixeltable.row            (one per inserted row, DEBUG level)
   └─ udf.<name>             (each UDF call, nested under its row)
```

Traces land in Grafana Cloud Traces (Tempo); the notebook also shows the metrics path
(`pixeltable.rows.written`) reaching Grafana.

In [ ]:
%pip install -qU 'pixeltable[otel]'

## A table with a UDF-backed computed column

The computed column runs `add_one` on every inserted row, producing the `udf.add_one` spans.

In [3]:
import pixeltable as pxt


@pxt.udf
def add_one(x: int) -> int:
    return x + 1


t = pxt.create_table('otel_demo1', {'c': pxt.Int}, if_exists='replace')
t.add_computed_column(inc=add_one(t.c))

Created table 'otel_demo1'.
Added 0 column values with 0 errors in 0.01 s


No rows affected.

## Insert

This is the traced operation — 10 rows (under the 100-span-per-operation cap, so every row gets a span).

In [5]:
status = t.insert([{'c': i} for i in range(10)])
status

Inserted 10 rows with 0 errors in 0.01 s (722.20 rows/s)


10 rows inserted.

## View it in Grafana

- **Traces**: Explore → your Traces (Tempo) data source → search Service Name `pixeltable`, or span name
  `pixeltable.insert`. Expand a trace to see `pixeltable.row` → `udf.add_one` nesting.
- **Metrics**: Explore → your Prometheus data source → query `pixeltable_rows_written_total`.

For more detail raise the span level (`hooks.set_span_level(hooks.TRACE)`); to reduce volume on a large
insert, leave it at the default INFO (operation + per-batch spans only).